<a href="https://colab.research.google.com/github/xuerongNanopay/ai_tut/blob/main/colab/LoRA_Llama_3_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip install trl
!pip install -U --force-reinstall bitsandbytes>=0.46.1
!pip uninstall -y torch torchvision torchaudio
!pip install --index-url https://download.pytorch.org/whl/cu128 torch torchvision torchaudio
!pip install tensorboard
# !pip uninstall -y numpy
# !pip install --no-cache-dir --force-reinstall "numpy<2.0"
!pip uninstall -y numpy pandas datasets pyarrow
!pip install --no-cache-dir --force-reinstall \
  "numpy<2.0" \
  "pandas<2.3" \
  "pyarrow" \
  "datasets"

In [ ]:
!pip list

In [ ]:
import transformers
print(transformers.__version__)
import bitsandbytes
print(bitsandbytes.__version__)

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
    TextStreamer,
    Trainer
)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import os, wandb
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
login(hf_token)

In [ ]:
torch.cuda.is_available()

In [ ]:
torch.cuda.device_count()

In [ ]:
model_name = "meta-llama/Llama-3.1-8B"
dataset_name = "scooterman/guanaco-llama3-1k"

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4", # qnf4 quant
    bnb_4bit_compute_dtype = torch.float16,
    bnb_4bit_use_double_quant = False,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    token=hf_token,
    device_map={"": 0}
)

model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
model.eval()

In [ ]:
dataset = load_dataset(dataset_name, split="train")

dataset["text"][0]

In [ ]:
def print_trainable_parameters(model):
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(f"Trained Params: {trainable_params} || Total Params: {all_param} || Trained/Total Percentage: {100*(trainable_params/all_param)}")

In [ ]:
rank = 8
peft_config = LoraConfig(
    r = rank,
    lora_alpha = rank*2,
    lora_dropout = 0.05,
    bias = "none",
    task_type ="CAUSAL_LM",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj"]
)

In [ ]:
os.environ["TENSORBOARD_LOGGING_DIR"] = "/content/logs"
training_arguments = SFTConfig(
    output_dir="/content/output",
    logging_steps = 20,
    report_to="tensorboard",
    num_train_epochs = 2,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 2,
    optim = "paged_adamw_8bit",
    save_steps = 100,
    learning_rate = 1e-4,
    weight_decay = 0.001,
    fp16 = False,
    bf16 = False,
    max_grad_norm = 0.3,
    max_steps = -1,
    warmup_steps = 0.3,
    lr_scheduler_type = "linear",
    dataset_text_field="text",
    packing=False
)

In [ ]:
trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    peft_config = peft_config,
    processing_class = tokenizer,
    args = training_arguments,
)

In [ ]:
trainer.train()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/output

In [ ]:
print_trainable_parameters(trainer.model)

In [ ]:
trainer.model.save_pretrained("/content/model/lora_model")

In [ ]:
model.eval()

In [ ]:
trainer.model.eval()

In [ ]:
def stream(user_input, model):
    system_prompt = 'Below is an instruction that describes a task. Write a response that appropriately complete'
    B_INST, E_INST = "### Instruction:\n", "### Response: \n"
    prompt = f"{system_prompt}{B_INST}{user_input.strip()}\n\n{E_INST}"
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    _ = model.generate(**inputs, streamer=streamer, max_new_tokens=128)

In [ ]:
stream("Tell me something about the Great Wall.", model)

In [ ]:
print(type(model))
print(type(trainer.model))
print(model is trainer.model)

## Use new model

In [ ]:
model_name = "meta-llama/Llama-3.1-8B"

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage = True,
    return_dict = True,
    torch_dtype = torch.float16,
    device_map={"": 0}
)


In [ ]:
base_model.eval()

In [ ]:
stream("Tell me something about the Great Wall.", base_model)